### Disclaimer
The dataset used in this project is based on data from a real organization however it has been modified and anonymized for confidentiality purposes. Company names, supplier information, plant identifiers, tracker details, and other potentially sensitive information have been removed, or altered to protect the identity of the organization and its business partners.

The analysis, business rules, and dashboard presented in this project are intended solely to demonstrate data analytics and reporting techniques acquired within the Data Analyics Course V3. 

### Project's information and objectives 

#### Company name: ABC 
  - Department: Business and Human Rights
  - Focus Area: Supplier's Audit Compliance and Risk monitoring 

The ABC’s Human Rights due Diligence (HRDD) program is designed to monitor high risk supplier plants, oversee audit compliance, and track the remediation of audit non-conformities. These non-conformities are categorized according to Hypercare levels, enabling the team to prioritize corrective actions and monitor supplier performance effectively.
This project aims to assess ABC Company's supplier plant data provided by the HRDD team.
The datasets provided by ABC serve as the foundation for this assessment and support the evaluation of supplier compliance and human rights risk management.
#### ABC’s Challenges and Solutions
For the reporting year of 2026, the HRDD team had a set a target of achieving a 98% audit completion rate across supplier plants.
Currently, the team manages multiple independent trackers to identify high-risk sites, monitor required audits, and review Hypercare cases. This fragmented approach makes it more difficult to obtain a consolidated view of supplier compliance, prioritize actions, and effectively monitor performance.

To address these challenges, this project consolidates HRDD data into an interactive dashboard that provides a single source of truth for supplier compliance. The solution enables the HRDD team to:
 - Monitor audit completion performance against the 98% target.
 - Identify high-risk supplier plants requiring audit.	
 - Track Hypercare status.
 - Prioritize outstanding actions and improve decision-making through real-time insights.
 - Reduce manual reporting and enhance operational efficiency.

#### Project's phases
This script is structured in 5 phases:
1. Data Preparation
   - Import Python libraries, and ABC's datasets. Ensure the data has been loaded correctly and is ready for analysis.
2. Data cleaning and transformation
   - Assess and prepare the datasets by resolving data quality issues, ensuring consistency across variables, standardizing formats, and applying necessary transformations to facilitate the analysis.
   Ensure to follow ABC's guidelines.
3. Data merge
   - Combine the cleaned datasets into a single dataframe.
4. Exploratory data analysis
   - Summarize the integrated dataset to identify the factors related to ABC's interests.
5. Final conclusions
   - Summarize the key findings and highlight the main insights.

### ABC Guidelines and Definitions
The analysis should apply the following criteria accross all Regions and Spend categories:

- For "In Trade", include only the following statuses: 
  - In trade 
  - Pending trade confirmation
  - Not in Supplier DB

- High risk plant definition: 
  - Membership Status: Active
  - SAQ Progress: 100% completed
  - Risk Category: High

- For high risk plants, the audit requirement is determined based on the audit date.
  - A plant is considered Compliant if it has been audited within the last three years, measured from the current year (2026).
  - A plant requires an audit if: **no audit record is available**, or the **audit was conducted more than three years ago**.

- Hypercare levels are assigned based on the number and severity of Non-Verified Non-Conformities (NCs) identified during supplier audits. Only supplier plants with a valid audit record are eligible for Hypercare classification. Plants without an audit must not be assigned a Hypercare level, as blank or zero values in the non-conformity fields may simply indicate that no audit has been performed.
  - Level 0 = 0 Non-Verified NCs (attention to empty cells or zeros appearing when no audit has been done)
  - Level 1 = 1 or more Non-Verified Minor NCs
  - Level 2 = 1 or more Non-Verified Major NCs
  - Level 3 = 1 or more Non-Verified Critical NCs
  - Level 4 = 1 or more Business Critical NCs, or a total of 25 or more Non-verified NCs 


**Phase1** - *Data Preparation*

This section contains the follow steps:
 - Import Python libraries 
 - Import datasets

In [107]:
# Import Python libraries 

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

sns.set_theme(style="whitegrid")

In [108]:
# Import datasets

import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")

suppliers_orginal = pd.read_excel(DATA_DIR / "Supplier base.xlsx")
suppliers = pd.read_excel(DATA_DIR / "Supplier base.xlsx")

hypercare_orginal = pd.read_csv(DATA_DIR / "Hypercare Analysis.csv",sep=None,engine="python")
hypercare = pd.read_csv(DATA_DIR / "Hypercare Analysis.csv",sep=None,engine="python")


**Phase 2:** *Data cleaning and transformation*

In this section, the below indicated actions will be applied for the two datasets provided by the ABC: **Supplier base** and **Hypercare Analysis**.

- Clean column names, and standardise columns where needed;
- Remove duplicates;
- Remove unnecessary columns;
- Handle missing values;
- Transform data, and create customised columns essential for client analysis:
    - High Risk
    - Audit Status
    - Audit Request
    - Hypercare Level

In [109]:
# Clean columns names and standardisation

def clean_headers(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace('\ufeff', '', regex=False)
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )
    return df


def clean_plant_code(df):
    df = df.copy()
    df["Plant code"] = (
        df["Plant code"]
        .astype(str)
        .str.strip()
        .str.upper()
    )
    return df

suppliers = clean_headers(suppliers)
hypercare = clean_headers(hypercare)

suppliers = clean_plant_code(suppliers)
hypercare = clean_plant_code(hypercare)

In [110]:
# Spend category format update
suppliers["Spend Category"] = suppliers["Spend Category"].replace({"Raw Materials": "Raw materials"})
suppliers["Spend Category"].value_counts()

Spend Category
Not in Supplier DB         2257
Raw materials              1775
Packaging materials         801
EM                          131
Licensee                     94
Co-Packing                   88
Indirect - out of scope       8
Name: count, dtype: int64

In [111]:
# Remove duplicates
suppliers = suppliers.drop_duplicates().copy()
hypercare = hypercare.drop_duplicates().copy()

In [112]:
# Verify data structure
print("Suppliers:", suppliers.shape)
print("Hypercare:", hypercare.shape)

Suppliers: (4930, 37)
Hypercare: (4888, 13)


**Dataset: Supplier base**

In [113]:
# Verify missing values and start cleaning decison making
suppliers.isnull().sum()

My Plants/My Suppliers' Plant    4930
Plant name                       4930
Plant code                          0
Supplier name                    4930
Supplier code                    4930
Customer Plant reference         4930
Customer name                    4930
Customer code                    4930
Customer supplier reference      4930
Business type                    1068
Primary Plant activity           1066
Trading relationship type        4930
Direct supplier name             4930
Direct supplier code             4930
Trading relationship code        4930
Trading relationship status      4930
SAQ visibility                   4930
Audit visibility                 4930
Date relationship initiated      4930
Initiated by                     4930
Membership status                   0
Membership expiry date           4930
Total workers                    4930
SAQ progress                     1470
SAQ last modified date           4930
Combined risk category           1257
Combined ris

*Observation:* It is noticed that there are several columns with nulls that would not be applicable for the final analysis and the client's request. 
The following actions consist in removing unnecessary columns.

In [114]:
# Remove all columns with nulls
suppliers.dropna(axis=1, how="all", inplace=True)

# Verify whatelse can be removed
suppliers.isnull().sum()

Plant code                   0
Business type             1068
Primary Plant activity    1066
Membership status            0
SAQ progress              1470
Combined risk category    1257
Combined risk score       1257
Audit filter              1381
Audit code                2147
Audit date                2147
Audit nature              2147
Audit schedule type       2147
Region                     572
BU                           0
Spend Category               0
In Trade?                    0
Phase                        0
dtype: int64

In [115]:
# Additional columns to remove

columns_to_remove = ["Business type", "Primary Plant activity", "Combined risk score", "Audit filter", 
                     "Audit nature", "Audit schedule type", "Phase"]

suppliers.drop(columns=columns_to_remove, inplace=True)

In [116]:
# Verify nulls under Region (key client's variable)
suppliers["Region"].value_counts(dropna=False)

Region
Not in Supplier DB    2206
MEU                    918
AMEA                   758
NaN                    572
LA                     476
Name: count, dtype: int64

In [117]:
suppliers["Region"].value_counts(dropna=False, normalize=True) * 100

Region
Not in Supplier DB    44.746450
MEU                   18.620690
AMEA                  15.375254
NaN                   11.602434
LA                     9.655172
Name: proportion, dtype: float64

*Observation:* Around 45% of the plants are classified as "Not in Supplier DB". ABC has not shared any context about this classification. For the purposes of this analysis, these records are treated as plants with no assigned Region, and will be retained in the dataset. 
Keeping these records ensures that any potential high risk plant is included in the assessment. If high risk sites are identified within this group, further internal investigation will be required to determine the appropriate regional ownership. Given the potential reputational and compliance risk associated with high risk suppliers, establishing clear accountability is critical to enable timely follow ups and actions needed. 

In addition, about 11% of the records have a null Region value. Before excluding these records from the analysis, similarly to the above scenario, it is important to verify whether any of them are classified as high risk. This validation helps ensure that potentially critical sites are not excluded from further assessment.

In [118]:
# Check High risk sites with no Region classified
# High risk = Combined risk category "High" and "100" SAQ progress 

critical_null_region = suppliers[(suppliers["Region"].isna()) &
                                 (suppliers["Combined risk category"] == "High") &
                                 (suppliers["SAQ progress"] == 100)]

print(f"Critical sites (High category & SAQ 100%) with null Region: {len(critical_null_region)}")

Critical sites (High category & SAQ 100%) with null Region: 77


*Observation:* A total of 77 High Risk plants do not have a Region assigned. To make sure these plants remain included in the final analysis, the missing Region values will be replaced with "No Region Classified". This keeps all identified High Risk plants in the dataset while clearly indicating that a regional assignment is still missing.

In [119]:
# Replace null values in Region
suppliers["Region"] = suppliers["Region"].fillna("No region classified")
suppliers["Region"].value_counts()

Region
Not in Supplier DB      2206
MEU                      918
AMEA                     758
No region classified     572
LA                       476
Name: count, dtype: int64

In [120]:
# Categorise high risk suppliers based on SAQ completion (100%) and High risk combined category
suppliers["High risk site"] = np.where((suppliers["SAQ progress"] == 100) &
                                       (suppliers["Combined risk category"] == "High"),
                                       "High Risk","Not applicable")

In [121]:
# Check distribution of High risk per Region 
pd.crosstab(suppliers["Region"],suppliers["High risk site"],margins=True)

High risk site,High Risk,Not applicable,All
Region,,,
AMEA,194,564,758
LA,103,373,476
MEU,123,795,918
No region classified,77,495,572
Not in Supplier DB,38,2168,2206
All,535,4395,4930


*Observation:* A total of 77 High Risk plants fall under "No region classified", while 38 High Risk plants are classified as "Not in Supplier DB". As these sites may represent a potential risk to the business, the ABC should consider investigating them internally to identify the appropriate regional ownership and determine the necessary follow-up actions. 

In [122]:
suppliers["In Trade?"].value_counts(dropna=False)

In Trade?
Not in Supplier DB            2206
In trade                      2067
Pending trade confirmation     268
No trade                       197
Out of scope                   142
HRDD Onboarding on hold         19
Out of scope - other            12
NOT TO BE COUNTED               11
Operation not started            3
Duplicate                        3
Duplicate- other                 2
Name: count, dtype: int64

*Observation:* Based on the ABC's guidelines, the analysis should only include plants with the following "In Trade" statuses: In trade; Pending trade confirmation; Not in Supplier DB.
All other statuses are considered out of scope for the analysis and will be excluded from the final dataset.
Before removing these records, an additional validation will be performed to identify any plants classified as High Risk within the excluded statuses. Since these sites may represent a potential risk to the business, they will be retained in a separate dataframe for the ABC's review and consideration.
Actions to apply moving foward:
- Create a separate dataframe containing all High Risk plants that belong to the excluded "In Trade" statuses.
- Save the dataframe for the client as a reference for further investigation and any required follow-up actions.
- Remove the excluded "In Trade" statuses from the main dataset.
- Continue the remaining analysis using only the focused "In Trade" statuses.

In [123]:
# Check "In Trade" statuses with High risk

pd.crosstab(suppliers["In Trade?"],suppliers["High risk site"],margins=True)

High risk site,High Risk,Not applicable,All
In Trade?,,,
Duplicate,0,3,3
Duplicate- other,0,2,2
HRDD Onboarding on hold,8,11,19
In trade,381,1686,2067
NOT TO BE COUNTED,2,9,11
No trade,38,159,197
Not in Supplier DB,38,2168,2206
Operation not started,1,2,3
Out of scope,17,125,142


In [124]:
status_counts = suppliers["In Trade?"].value_counts()
status_counts.loc[["No trade",
                   "Out of scope",
                   "HRDD Onboarding on hold",
                   "Out of scope - other",
                   "NOT TO BE COUNTED",
                   "Operation not started",
                   "Duplicate",
                   "Duplicate- other"]].sum()

np.int64(389)

In [125]:
# In Trade statuses to keep
keep_statuses = ["In trade","Pending trade confirmation","Not in Supplier DB"]

# Excluded statuses
client_considerations = suppliers[(~suppliers["In Trade?"].isin(keep_statuses)) &
                                  (suppliers["High risk site"] == "High Risk")]

print(f"High Risk plants for client consideration: {len(client_considerations)}")

client_considerations = client_considerations[["Plant code", "Region", "In Trade?", "High risk site"]]



High Risk plants for client consideration: 71


*Observation:* A total of 389 records fall outside the "In Trade" statuses and will be excluded from the analysis moving forward. A separate dataframe containing the 71 High Risk plants identified within these excluded statuses will be retained for the ABC's consideration.

In [126]:
# Client considerations report

client_considerations.to_excel(Path(DATA_DIR) / "Client_Considerations_High_Risk_Out_of_Scope.xlsx",
                               index=False)

In [127]:
# Remore unnecessary in trade status
# In Trade statuses to keep
keep_statuses = ["In trade","Pending trade confirmation","Not in Supplier DB"]

# Keep only the approved statuses
suppliers= suppliers[suppliers["In Trade?"].isin(keep_statuses)].copy()
suppliers["In Trade?"].value_counts()

In Trade?
Not in Supplier DB            2206
In trade                      2067
Pending trade confirmation     268
Name: count, dtype: int64

In [128]:
# Keep only active membership 
suppliers = suppliers[suppliers["Membership status"] != "Lapsed"]

In [129]:
# Category for Audit status based on High risk sites identified and the audit done in the past 3 year
# Reporting year
reporting_year = 2026
cutoff_date = pd.Timestamp(f"{reporting_year - 3}-01-01")

# Audit date is a datetime
suppliers["Audit date"] = pd.to_datetime(suppliers["Audit date"],errors="coerce")

suppliers["Audit Status"] = np.select([suppliers["High risk site"].eq("Not applicable"),
        suppliers["Audit date"].isna(),suppliers["Audit date"] >= cutoff_date],
        ["Not applicable",
        "No audit done",
        "Audit compliant"],
    default="Audit past due")

In [130]:
# Triger audit request for high risk with no audit done and audit past due 

def audit_request(status):

    if status == "Audit compliant":
        return "Audit Compliant"

    elif status in ["No audit done", "Audit past due"]:
        return "Request New Audit"

    else:
        return "Not applicable"


suppliers["Audit Request"] = (suppliers["Audit Status"]
    .apply(audit_request))

In [131]:
suppliers["Plant code"].duplicated().sum()

np.int64(0)

**Dataset: Hypercare Analysis**

In [132]:
hypercare.isnull().sum()

Plant code                               0
Primary Plant activity                1068
Audit code                            2136
Audit date                            2136
Audit nature                          2136
Audit schedule type                   2136
Non-verified NCs                      2136
Non-verified Business Critical NCs    2136
Non-verified Critical NCs             2136
Non-verified Major NCs                2136
Non-verified Minor NCs                2136
Overdue non-verified NCs              2136
Country/Region                           1
dtype: int64

In [133]:
# Remove unnecessary columns
hypercare = hypercare.drop_duplicates().copy()

columns_to_remove_hpc = ["Primary Plant activity","Audit nature", "Audit schedule type",
                         "Overdue non-verified NCs"]

hypercare.drop(columns=columns_to_remove_hpc, inplace=True)


In [134]:
# Remove null row under Country/Region
hypercare = hypercare.dropna(subset=["Country/Region"]).copy()

In [135]:
# Categorise Hypercare level based on Non-verified NCs and Non-verified NCs per it criticality
# See client's guidance for Hypercare level categorization

conditions = [
    # No audit information
    hypercare["Non-verified NCs"].isna(),

    # Hypercare
    (hypercare["Non-verified NCs"] >= 25) |
    (hypercare["Non-verified Business Critical NCs"] >= 1),

    # Level 3
    hypercare["Non-verified Critical NCs"] >= 1,

    # Level 2
    hypercare["Non-verified Major NCs"] >= 1,

    # Level 1
    hypercare["Non-verified Minor NCs"] >= 1,

    # Level 0
    hypercare["Non-verified NCs"] == 0
]

choices = [
    "No audit information",
    "Hypercare",
    "Level 3",
    "Level 2",
    "Level 1",
    "Level 0"
]

hypercare["Hypercare Level"] = np.select(conditions,choices,default="Review")

# Check review status 
hypercare[hypercare["Hypercare Level"] == "Review"]

,Plant code,Audit code,Audit date,Non-verified NCs,Non-verified Business Critical NCs,Non-verified Critical NCs,Non-verified Major NCs,Non-verified Minor NCs,Country/Region,Hypercare Level


*Observation:* Hypercare Level Classification: Plants are classified according to the client's business rules. Records with no audit information are identified first. Plants meeting the Hypercare criteria (≥25 non-verified NCs and at least one Business Critical NC) are prioritized over all other levels. The remaining plants are then classified based on the highest severity of outstanding non-verified NCs (Critical, Major, Minor), while plants with no outstanding NCs are assigned Level 0.

**3. Merge Datasets**
- Combine datasets into Risk_analysis
- Apply additional cleaning if needed

In [136]:
# Verify 
print("Suppliers duplicates:", suppliers["Plant code"].duplicated().sum())
print("Hypercare duplicates:", hypercare["Plant code"].duplicated().sum())

Suppliers duplicates: 0
Hypercare duplicates: 0


In [137]:
# Merge and verify if in "Plant code" there is any duplicate
risk_analysis= suppliers.merge(hypercare, on="Plant code", how="left")
risk_analysis["Plant code"].duplicated().sum()

np.int64(0)

In [138]:
# Check if Hypercare is popuated properly
risk_analysis["Hypercare Level"].value_counts(dropna=False)

Hypercare Level
Level 0                 1468
No audit information     870
Level 2                  530
Level 3                  374
Level 1                   81
NaN                       50
Hypercare                 21
Name: count, dtype: int64

**Observation:** It is noticed 50 nulls under Hypercare level. This might be due to the merger. In this case, it is ok to populate the information as No audit information. 

In [139]:
# Fill as "No audit information" for nulls under Hypercare Level 
risk_analysis["Hypercare Level"] = risk_analysis["Hypercare Level"].fillna("No audit information")
risk_analysis["Hypercare Level"].value_counts(dropna=False)

Hypercare Level
Level 0                 1468
No audit information     920
Level 2                  530
Level 3                  374
Level 1                   81
Hypercare                 21
Name: count, dtype: int64

In [140]:
# Verify if Audit status and audit request match
risk_analysis["Audit Request"].value_counts(dropna=False)

Audit Request
Not applicable       2930
Audit Compliant       441
Request New Audit      23
Name: count, dtype: int64

In [141]:
risk_analysis["Audit Status"].value_counts(dropna=False)

Audit Status
Not applicable     2930
Audit compliant     441
No audit done        17
Audit past due        6
Name: count, dtype: int64

In [142]:
# Identify High risk sites per Audit request status and Hypercare level 
pd.crosstab(index=[risk_analysis["High risk site"], risk_analysis["Audit Request"]],
            columns=risk_analysis["Hypercare Level"],margins=True)

Hypercare Level                   Hypercare  Level 0  Level 1  Level 2  \
High risk site Audit Request                                             
High Risk      Audit Compliant           18       80        2       48   
               Request New Audit          0        1        0        2   
Not applicable Not applicable             3     1387       79      480   
All                                      21     1468       81      530   

Hypercare Level                   Level 3  No audit information   All  
High risk site Audit Request                                           
High Risk      Audit Compliant        280                    13   441  
               Request New Audit        2                    18    23  
Not applicable Not applicable          92                   889  2930  
All                                   374                   920  3394

*Observation*: The above table gives an important insight showing that Hypercare is not limited to supplier plants classified as High Risk. A small number of plants outside the High Risk population have been assigned a Hypercare level, demonstrating that significant audit NCs may also exist beyond the defined High Risk criteria. Consequently, Hypercare should be treated as a complementary risk indicator when assessing supplier priorities.

**4. Exploratory Analysis**

- Overall High-Risk Profile
    - Total sites
    - High-Risk sites (%)
    - Not applicable sites - Low and Medium risk plants (%) 
    - Distribution by Region, and Spend Category
- Audit Status
    - High risk with audit complaint
    - High risk with audit required
    - Distribution by Region, and Spend Category
- Hypercare Analysis
    - Overall Hypercare Levels
    - Distribution by Region, and Spend Category

In [143]:
# Overall 
# Total unique sites
total_sites = risk_analysis["Plant code"].nunique()

# Not applicable sites (Low and Medium risk)
not_applicable_sites = risk_analysis.loc[risk_analysis["High risk site"] == "Not applicable",
                                         "Plant code"].nunique()

# High-risk sites
high_risk_sites = risk_analysis.loc[
    risk_analysis["High risk site"] == "High Risk","Plant code"].nunique()

# High-risk sites requiring an audit
audit_required = risk_analysis.loc[risk_analysis["Audit Request"] == "Request New Audit",
                                   "Plant code"].nunique()

# High-risk sites that are audit compliant
audit_compliant = risk_analysis.loc[risk_analysis["Audit Status"] == "Audit compliant",
                                    "Plant code"].nunique()

# Percentages
not_applicable_pct = (not_applicable_sites / total_sites) * 100
high_risk_pct = (high_risk_sites / total_sites) * 100
audit_required_pct = (audit_required / high_risk_sites) * 100
audit_compliant_pct = (audit_compliant / high_risk_sites) * 100

# Display KPIs
print(f"Total Sites: {total_sites}")
print(f"Low and Medium risk Sites: {not_applicable_sites} ({not_applicable_pct:.1f}%)")
print(f"High-Risk Sites: {high_risk_sites} ({high_risk_pct:.1f}%)")
print(f"Audit Compliant (High-Risk): {audit_compliant} ({audit_compliant_pct:.1f}%)")
print(f"Audit Required (High-Risk): {audit_required} ({audit_required_pct:.1f}%)")

Total Sites: 3394
Low and Medium risk Sites: 2930 (86.3%)
High-Risk Sites: 464 (13.7%)
Audit Compliant (High-Risk): 441 (95.0%)
Audit Required (High-Risk): 23 (5.0%)


In [144]:
# High Risk Sites by Region
region_summary = (risk_analysis.groupby("Region")
                  .agg(Total_Sites=("Plant code", "nunique"),
                       High_Risk_Sites=("High risk site",
                                        lambda x: (x == "High Risk").sum())))

region_summary["High Risk %"] = (region_summary["High_Risk_Sites"] / 
                                 region_summary["Total_Sites"] * 100).round(1)

region_summary.sort_values("High Risk %", ascending=False)

,Total_Sites,High_Risk_Sites,High Risk %
Region,,,
AMEA,656,174,26.5
LA,366,78,21.3
MEU,789,111,14.1
No region classified,458,63,13.8
Not in Supplier DB,1125,38,3.4


In [145]:
# High Risk by Spend Category

spend_summary = (risk_analysis.groupby("Spend Category")
                 .agg(Total_Sites=("Plant code", "nunique"),
                      High_Risk_Sites=("High risk site",
                                       lambda x: (x == "High Risk").sum())))

spend_summary["High Risk %"] = (spend_summary["High_Risk_Sites"] / 
                                spend_summary["Total_Sites"] * 100).round(1)

spend_summary.sort_values("High Risk %", ascending=False)

,Total_Sites,High_Risk_Sites,High Risk %
Spend Category,,,
EM,113,34,30.1
Licensee,78,15,19.2
Packaging materials,692,133,19.2
Raw materials,1314,232,17.7
Co-Packing,72,12,16.7
Not in Supplier DB,1125,38,3.4


In [146]:
# Total Audit Status of High-Risk Sites
high_risk_df = risk_analysis[risk_analysis["High risk site"] == "High Risk"].copy()

audit_summary = (high_risk_df["Audit Status"].value_counts().rename_axis("Audit Status")
                 .reset_index(name="Sites"))

audit_summary["Percentage"] = (audit_summary["Sites"] /
                               audit_summary["Sites"].sum() * 100).round()

audit_summary

,Audit Status,Sites,Percentage
0,Audit compliant,441,95.0
1,No audit done,17,4.0
2,Audit past due,6,1.0


**Observation:** 95% of high risk plants had done and audit in the past 3 years, indicating a positive compliance status. Only 23 high risk plants would require a new audit. 

In [147]:
# Audit required by Region

audit_region = pd.crosstab(high_risk_df["Region"],high_risk_df["Audit Request"],margins=True)
audit_region

Audit Request,Audit Compliant,Request New Audit,All
Region,,,
AMEA,164,10,174
LA,75,3,78
MEU,106,5,111
No region classified,60,3,63
Not in Supplier DB,36,2,38
All,441,23,464


**Observation:** Of the 23 High Risk plants requiring a new audit, **5** have not been assigned to a Region. ABC should investigate these plants internally to identify the appropriate regional ownership. This will help ensure that all High Risk plants are assigned to the correct team and receive the necessary follow-up, reducing potential business and reputational risks.

The same approach could also be applied to the Audit Compliant plants that have no assigned Region. Although these plants do not require immediate action, assigning them to the appropriate Region would improve data quality and support more accurate reporting and future analysis.
 

In [148]:
# Audit compliance % per Region 

audit_region_pct = pd.crosstab(high_risk_df["Region"],high_risk_df["Audit Request"],
                               normalize="index").mul(100).round()

audit_region_pct

Audit Request,Audit Compliant,Request New Audit
Region,,
AMEA,94.0,6.0
LA,96.0,4.0
MEU,95.0,5.0
No region classified,95.0,5.0
Not in Supplier DB,95.0,5.0


**Observation:** Considering the **98% compliance target**, the overall results and the performance across all Regions indicate that the client is well positioned to achieve this objective. However, it is important to identify and assign the plants with **no regional classification**, as they are not currently attributed to any Region and could affect the overall compliance target.


In [149]:
# Total per spend category 
audit_total = pd.crosstab(high_risk_df["Spend Category"],high_risk_df["Audit Request"])
audit_total 

Audit Request,Audit Compliant,Request New Audit
Spend Category,,
Co-Packing,12,0
EM,34,0
Licensee,15,0
Not in Supplier DB,36,2
Packaging materials,128,5
Raw materials,216,16


In [150]:
# Audit compliance % per spend category 
audit_spend_pct = pd.crosstab(high_risk_df["Spend Category"],high_risk_df["Audit Request"],
                              normalize="index").mul(100).round()

audit_spend_pct

Audit Request,Audit Compliant,Request New Audit
Spend Category,,
Co-Packing,100.0,0.0
EM,100.0,0.0
Licensee,100.0,0.0
Not in Supplier DB,95.0,5.0
Packaging materials,96.0,4.0
Raw materials,93.0,7.0


Observation: Considering the compliance target by Spend Category, Raw Materials and Packaging Materials have not yet reached the 98% target. It should also be noted that some plants are currently classified as "Not in Supplier DB" and therefore have no assigned Spend Category. Identifying and assigning these plants to the appropriate Spend Category may affect the compliance results and provide a more accurate view of performance across all categories.

In [151]:
# 4.1 Overall Hypercare Level Distribution

hypercare_summary = (risk_analysis["Hypercare Level"].value_counts()
                     .rename_axis("Hypercare Level")
                     .reset_index(name="Sites"))

hypercare_summary["Percentage"] = (hypercare_summary["Sites"] /
                                   hypercare_summary["Sites"].sum() * 100).round(1)

hypercare_summary

,Hypercare Level,Sites,Percentage
0,Level 0,1468,43.3
1,No audit information,920,27.1
2,Level 2,530,15.6
3,Level 3,374,11.0
4,Level 1,81,2.4
5,Hypercare,21,0.6


In [152]:
# Total Hypercare per region
hypercare_region = pd.crosstab(risk_analysis["Region"],
                               risk_analysis["Hypercare Level"],margins=True)

hypercare_region

Hypercare Level,Hypercare,Level 0,Level 1,Level 2,Level 3,No audit information,All
Region,,,,,,,
AMEA,7,344,6,94,105,100,656
LA,4,145,9,58,67,83,366
MEU,5,395,18,124,90,157,789
No region classified,5,165,24,106,62,96,458
Not in Supplier DB,0,419,24,148,50,484,1125
All,21,1468,81,530,374,920,3394


In [153]:
# Hypercare per region %
hypercare_region_pct = (pd.crosstab(risk_analysis["Region"],
                                    risk_analysis["Hypercare Level"], normalize="index") * 100).round(1)

hypercare_region_pct

Hypercare Level,Hypercare,Level 0,Level 1,Level 2,Level 3,No audit information
Region,,,,,,
AMEA,1.1,52.4,0.9,14.3,16.0,15.2
LA,1.1,39.6,2.5,15.8,18.3,22.7
MEU,0.6,50.1,2.3,15.7,11.4,19.9
No region classified,1.1,36.0,5.2,23.1,13.5,21.0
Not in Supplier DB,0.0,37.2,2.1,13.2,4.4,43.0


In [154]:
# Hypercare per spend 
hypercare_spend = pd.crosstab(risk_analysis["Spend Category"],
                              risk_analysis["Hypercare Level"], margins=True)

hypercare_spend.sort_values("Hypercare", ascending= False)

Hypercare Level,Hypercare,Level 0,Level 1,Level 2,Level 3,No audit information,All
Spend Category,,,,,,,
All,21,1468,81,530,374,920,3394
Raw materials,8,634,33,218,174,247,1314
EM,6,48,3,25,29,2,113
Packaging materials,6,303,12,106,91,174,692
Licensee,1,33,3,17,18,6,78
Co-Packing,0,31,6,16,12,7,72
Not in Supplier DB,0,419,24,148,50,484,1125


In [155]:
hypercare_spend_pct = (pd.crosstab(risk_analysis["Spend Category"],
                                   risk_analysis["Hypercare Level"],normalize="index") * 100).round(1)

hypercare_spend_pct.sort_values("Hypercare", ascending= False)

Hypercare Level,Hypercare,Level 0,Level 1,Level 2,Level 3,No audit information
Spend Category,,,,,,
EM,5.3,42.5,2.7,22.1,25.7,1.8
Licensee,1.3,42.3,3.8,21.8,23.1,7.7
Packaging materials,0.9,43.8,1.7,15.3,13.2,25.1
Raw materials,0.6,48.2,2.5,16.6,13.2,18.8
Co-Packing,0.0,43.1,8.3,22.2,16.7,9.7
Not in Supplier DB,0.0,37.2,2.1,13.2,4.4,43.0


PowerBI data preparetion

In [156]:
# Create a copy of the final dataset for PowerBI
powerbi_data = risk_analysis.copy()

In [ ]:
powerbi_data = risk_analysis[
    [
        "Plant code",
        "Region",
        "Spend Category",
        "Combined risk category",
        "High risk site",
        "Audit Status",
        "Audit Request",
        "Hypercare Level"
    ]
].copy()


In [ ]:
# Export to CSV
powerbi_data.to_csv("PowerBI_Data.csv",index=False)

### Final Conclusions

This project successfully integrated multiple HRDD datasets to create a consolidated view of supplier audit compliance and Hypercare status. By applying ABC's business rules, the analysis identified the population of High Risk supplier plants, assessed audit compliance, and highlighted suppliers requiring further attention.

The results indicate that ABC is well positioned to achieve its 98% audit compliance target, with only 23 High Risk plants identified as requiring a new audit. While overall compliance is strong, the analysis also showed that supplier risk extends beyond audit completion. The analysis demonstrated that Hypercare should be considered a complementary risk indicator, as a small number of supplier plants outside the High Risk definition were also assigned Hypercare levels. This reinforces the importance of monitoring both audit compliance and outstanding non-conformities to obtain a more complete view of supplier risk.

Finally, the interactive dashboard developed as part of this project provides the HRDD team with a centralized view of supplier performance, enabling more effective monitoring of audit compliance, prioritization of remediation activities, and data-driven decision-making. The solution reduces reliance on multiple manual trackers and supports a more efficient and proactive supplier risk management process.